# Model Fine-tuning with Hugging Face
This notebook demonstrates how to fine-tune a pre-trained model on your custom dataset.

In [ ]:
# Install required packages
%pip install transformers datasets torch evaluate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import torch
import evaluate
import numpy as np

In [ ]:
# Load your dataset from Hugging Face Hub
# Replace 'dataset_name' with your actual dataset
dataset = load_dataset('samhog/psychology-6k')

# Load pre-trained model and tokenizer
# Replace 'model_name' with your chosen model
model_name = "deepseek-ai/DeepSeek-V3-0324"  # Example model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

# Start training
trainer.train()

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")